In [ ]:
!pip install opencv-python scikit-image scikit-learn tensorflow matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

ModuleNotFoundError: No module named 'cv2'

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print("Train:", x_train.shape, y_train.shape)
print("Test: ", x_test.shape, y_test.shape)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img, lbl in zip(axes.ravel(), x_train[:16], y_train[:16]):
    ax.imshow(img, cmap="gray")
    ax.set_title(lbl)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    results[name] = acc
    print(f"{name} — test accuracy: {acc:.4f}\n")
    print(classification_report(y_true, y_pred, digits=4))
    fig, ax = plt.subplots(figsize=(6, 6))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} — confusion matrix")
    plt.show()

In [ ]:
def get_contours(img):
    _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    # [-2:] keeps this working on both OpenCV 3 and OpenCV 4
    contours, hierarchy = cv2.findContours(binary, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_NONE)[-2:]
    return binary, contours, hierarchy

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for ax, img, lbl in zip(axes.ravel(), x_train[:16], y_train[:16]):
    _, contours, hierarchy = get_contours(img)
    canvas = cv2.resize(cv2.cvtColor(img, cv2.COLOR_GRAY2RGB), (112, 112),
                        interpolation=cv2.INTER_NEAREST)
    for i, c in enumerate(contours):
        is_hole = hierarchy[0][i][3] != -1
        color = (255, 0, 0) if is_hole else (0, 255, 0)
        cv2.drawContours(canvas, [c * 4 + 2], -1, color, 1)
    ax.imshow(canvas)
    ax.set_title(lbl)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
N_POINTS = 20  # points sampled along the main contour

def contour_features(img):
    _, contours, hierarchy = get_contours(img)
    feats = np.zeros(10 + 7 + 2 * N_POINTS, dtype=np.float32)
    if len(contours) == 0:
        return feats

    outer_idx = [i for i in range(len(contours)) if hierarchy[0][i][3] == -1]
    hole_idx  = [i for i in range(len(contours)) if hierarchy[0][i][3] != -1]

    # Main (largest) outer contour
    main = max((contours[i] for i in outer_idx), key=cv2.contourArea)
    area  = cv2.contourArea(main)
    perim = cv2.arcLength(main, True)
    x, y, w, h = cv2.boundingRect(main)
    hull_area = cv2.contourArea(cv2.convexHull(main))

    # Holes (ignore tiny noise holes)
    holes = [contours[i] for i in hole_idx if cv2.contourArea(contours[i]) > 2]
    hole_area = sum(cv2.contourArea(c) for c in holes)
    if holes:
        cys = []
        for c in holes:
            M = cv2.moments(c)
            if M["m00"] > 0:
                cys.append((M["m01"] / M["m00"] - y) / max(h, 1))
        hole_cy = np.mean(cys) if cys else -1
    else:
        hole_cy = -1

    # 1) Geometric shape descriptors
    feats[0] = area
    feats[1] = perim
    feats[2] = w / max(h, 1)                          # aspect ratio
    feats[3] = area / max(w * h, 1)                   # extent
    feats[4] = area / max(hull_area, 1e-6)            # solidity
    feats[5] = 4 * np.pi * area / max(perim**2, 1e-6) # circularity
    feats[6] = len(holes)                             # number of holes
    feats[7] = hole_area / max(area, 1e-6)            # hole size ratio
    feats[8] = len(outer_idx)                         # number of separate pieces
    feats[9] = hole_cy                                # vertical hole position (6 vs 9)

    # 2) Hu moments (log-scaled)
    hu = cv2.HuMoments(cv2.moments(main)).flatten()
    feats[10:17] = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)

    # 3) Shape signature: N points along the contour, normalized to the bounding box
    idx = np.linspace(0, len(main) - 1, N_POINTS).astype(int)
    pts = main[idx, 0, :].astype(np.float32)
    pts = (pts - [x, y]) / [max(w, 1), max(h, 1)]
    feats[17:] = pts.flatten()
    return feats

print("Feature length:", len(contour_features(x_train[0])))

In [ ]:
t0 = time.time()
X_train_cnt = np.array([contour_features(img) for img in x_train])
X_test_cnt  = np.array([contour_features(img) for img in x_test])
print("Train features:", X_train_cnt.shape, "| Test features:", X_test_cnt.shape)
print(f"Extraction time: {time.time() - t0:.1f}s")

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
rf.fit(X_train_cnt, y_train)
print(f"Training time: {time.time() - t0:.1f}s")

evaluate("Contours + RandomForest", y_test, rf.predict(X_test_cnt))

In [ ]:
img = x_train[0]
feat, hog_img = hog(img, orientations=9, pixels_per_cell=(7, 7),
                    cells_per_block=(2, 2), block_norm="L2-Hys", visualize=True)
print("HOG feature length:", feat.shape[0])

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img, cmap="gray");     axes[0].set_title(f"Digit {y_train[0]}")
axes[1].imshow(hog_img, cmap="gray"); axes[1].set_title("HOG")
for a in axes: a.axis("off")
plt.show()

In [ ]:
def hog_features(images):
    return np.array([
        hog(img, orientations=9, pixels_per_cell=(7, 7),
            cells_per_block=(2, 2), block_norm="L2-Hys")
        for img in images
    ], dtype=np.float32)

t0 = time.time()
X_train_hog = hog_features(x_train)
X_test_hog  = hog_features(x_test)
print("Train:", X_train_hog.shape, "| Test:", X_test_hog.shape)
print(f"Extraction time: {time.time() - t0:.1f}s")

In [ ]:
N_TRAIN = 20000   # set to 60000 for the best accuracy (training gets much slower)

svm = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale"))

t0 = time.time()
svm.fit(X_train_hog[:N_TRAIN], y_train[:N_TRAIN])
print(f"Training time: {time.time() - t0:.1f}s")

evaluate("HOG + SVM", y_test, svm.predict(X_test_hog))

In [ ]:
x_train_cnn = (x_train.astype("float32") / 255.0)[..., np.newaxis]
x_test_cnn  = (x_test.astype("float32") / 255.0)[..., np.newaxis]
print(x_train_cnn.shape, x_test_cnn.shape)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3,
                                           restore_best_weights=True)
t0 = time.time()
history = model.fit(x_train_cnn, y_train,
                    epochs=15, batch_size=128,
                    validation_split=0.1,
                    callbacks=[early_stop], verbose=1)
print(f"Training time: {time.time() - t0:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.show()

In [ ]:
y_pred_cnn = model.predict(x_test_cnn, verbose=0).argmax(axis=1)
evaluate("CNN", y_test, y_pred_cnn)

In [ ]:
names = list(results.keys())
accs = [results[n] * 100 for n in names]

plt.figure(figsize=(7, 4))
bars = plt.bar(names, accs, color=["#e07a5f", "#3d85c6", "#81b29a"])
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, a + 0.3, f"{a:.2f}%", ha="center")
plt.ylim(min(accs) - 5, 100.5)
plt.ylabel("Test accuracy (%)")
plt.title("MNIST: Contours+RF vs HOG+SVM vs CNN")
plt.show()

for n, a in zip(names, accs):
    print(f"{n:28s} {a:.2f}%")